In [2]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

PARCELS_DIR = DATA_DIR / "parcels" / "processed"

parcels_in_toc_path = PARCELS_DIR / "parcels_assessor_toc.gpkg"

parcels_in_toc_path

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/parcels_assessor_toc.gpkg')

In [3]:
parcels_in_toc = gpd.read_file(parcels_in_toc_path)

print(parcels_in_toc.columns)
print(len(parcels_in_toc))

Index(['APN', 'Floor', 'LandSize', 'StartDate', 'Shp_Area', 'Shp_Length',
       'PARCEL_NO', 'PROPERTYUSECODE', 'FULLCASHVALUE', 'LANDFULLCASHVALUE',
       'IMPRFULLCASHVALUE', 'LPVAMOUNT', 'LPVASSESSEDVALUE', 'SQFOOTAGE',
       'NUMBEROFUNITS', 'NEIGHBORHOODCODE', 'MARKETAREACODE', 'index_right',
       'APPLICABIL', 'geometry'],
      dtype='object')
1736206


In [4]:
print(parcels_in_toc.crs)

EPSG:2868


In [6]:
parcels = parcels_in_toc.copy()

# Area in acres from geometry (sq ft / 43,560)
parcels["area_acres"] = parcels.geometry.area / 43560.0

# Drop anything with zero/negative area just in case
parcels = parcels[parcels["area_acres"] > 0]

In [7]:
toc_parcels = parcels[parcels["APPLICABIL"].notna()].copy()
print(len(toc_parcels))

49390


In [8]:
group_cols = ["index_right", "APPLICABIL"]

toc_stats = (
    toc_parcels
    .groupby(group_cols, as_index=False)
    .agg(
        total_taxable_value=("LPVASSESSEDVALUE", "sum"),
        total_full_cash=("FULLCASHVALUE", "sum"),
        total_land_value=("LANDFULLCASHVALUE", "sum"),
        total_acres=("area_acres", "sum"),
        parcel_count=("APN", "count"),
    )
)

toc_stats["taxable_value_per_acre"] = (
    toc_stats["total_taxable_value"] / toc_stats["total_acres"]
)

toc_stats.head()

,index_right,APPLICABIL,total_taxable_value,total_full_cash,total_land_value,total_acres,parcel_count,taxable_value_per_acre
0,0.0,TOD District - Gateway,285984758.0,3.718919e+09,8.830583e+08,2346.119575,3259,121896.923347
1,1.0,TOD District - Eastlake Garfield,207168885.0,3.014334e+09,8.307006e+08,1015.214786,3323,204064.093422
2,2.0,TOD District - Midtown,468187974.0,7.310771e+09,1.939664e+09,1053.891243,4552,444246.953429
3,3.0,TOD District - Uptown,171863623.0,3.589014e+09,9.414611e+08,1040.744605,3746,165135.252405
4,4.0,TOD District - Solano,91072483.0,1.978354e+09,4.115623e+08,848.835227,3414,107291.120997


In [9]:
out_csv_path = PROJECT_ROOT / "outputs" / "toc_land_value_stats.csv"
toc_stats.to_csv(out_csv_path, index=False)